# PCB 결함 검사 — YOLOv8 하이퍼파라미터 튜닝 Colab 노트북

**Team Convex | ATI 인턴 프로젝트**

이 노트북은 `scripts/run_tune.bat`과 **동일한 튜닝 흐름**을 Google Colab에서 실행하며,
Google Drive 연동으로 **세션이 끊겨도 결과가 Drive에 안전하게 보존**됩니다.

> **유전 알고리즘(GA) 기반 Ultralytics 내장 튜너를 사용합니다.**  
> 튜닝은 일반 학습보다 훨씬 오랜 시간이 소요됩니다 (`iterations × epochs` 단위).

---

## 실행 전 준비

1. **GitHub Token 등록** (Private Repo 접근용)
   - Colab 왼쪽 사이드바 🔑 아이콘 클릭
   - `GITHUB_TOKEN` 이름으로 Personal Access Token(PAT) 추가
   - PAT 생성: GitHub → Settings → Developer settings → Personal access tokens
   - 필요 권한: `Contents: Read`

2. **dataset.zip Drive 업로드**
   - Google Drive `MyDrive/pcb-defect-inspection/` 폴더에 `dataset.zip` 업로드

3. **런타임 유형 확인**
   - 메뉴: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

---

## 학습 흐름 (run_tune.bat과 동일)

| 셀 | 단계 | 비고 |
|----|------|------|
| ① | 환경 설정 (GPU 확인) | 경로 상수 정의 |
| ② | Google Drive 마운트 | |
| ③ | Repo 클론 & 패키지 설치 | Private repo 지원 |
| ④ | 설정 확인 | tune 파라미터 출력 |
| ⑤ | 전처리 | 이미 완료 시 자동 스킵 |
| ⑥ | 이전 튜닝 결과 체크 | Drive의 runs/tune 자동 감지 |
| ⑦ | 튜닝 실행 | src/tune.py 실행 |
| ⑧ | Drive 동기화 | 세션 종료 전 필수! |
| ⑨ | 결과 확인 | best_hyperparameters.yaml 출력 (선택 사항) |

## ① 환경 설정

> `DRIVE_PROJECT_DIR`이 실제 Drive 폴더와 다르면 수정 후 실행하세요.

In [ ]:
# ============================================================
# ① 환경 설정 — 아래 상수를 확인 후 실행하세요
# ============================================================

# --- GitHub 설정 ---
GITHUB_REPO   = "june0922/pcb-defect-inspection"
GITHUB_BRANCH = "main"  # 사용할 브랜치명
PROJECT_DIR   = "/content/pcb-defect-inspection"

# --- Google Drive 경로 ---
# dataset.zip을 업로드한 Drive 폴더 경로 (필요시 수정)
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/pcb-defect-inspection"
DRIVE_DATASET_ZIP = f"{DRIVE_PROJECT_DIR}/dataset.zip"

# --- GPU 확인 ---
import subprocess
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f"✅ GPU: {r.stdout.strip()}")
else:
    print("⚠️  GPU 없음 — 런타임 유형을 'GPU'로 변경하세요.")
    print("   메뉴: 런타임 > 런타임 유형 변경 > T4 GPU 선택")

print(f"\n프로젝트 디렉토리 : {PROJECT_DIR}")
print(f"Drive 프로젝트 폴더: {DRIVE_PROJECT_DIR}")
print(f"Drive dataset.zip  : {DRIVE_DATASET_ZIP}")

## ② Google Drive 마운트

In [ ]:
# ============================================================
# ② Google Drive 마운트
# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print("✅ Drive 마운트 완료")
print(f"Drive 프로젝트 폴더: {DRIVE_PROJECT_DIR}")
contents = os.listdir(DRIVE_PROJECT_DIR)
print(f"현재 내용: {contents if contents else '(비어 있음)'}")

## ③ Repo 클론 & 패키지 설치 (Private Repo)

> **사전 준비**: Colab 왼쪽 🔑 아이콘 → `GITHUB_TOKEN` 등록 필요

In [ ]:
# ============================================================
# ③ Repo 클론 & 패키지 설치 (Private Repo)
#    보안: subprocess로 클론해 토큰이 출력에 노출되지 않습니다.
# ============================================================
from google.colab import userdata
import subprocess, os

# --- GitHub Token 로드 ---
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("✅ GitHub Token 로드됨 (Colab Secrets)")
except Exception:
    GITHUB_TOKEN = ""
    print("❌ GITHUB_TOKEN을 찾을 수 없습니다.")
    print("   해결: 왼쪽 사이드바 키 콤 클릭 → 'GITHUB_TOKEN' 추가")
    raise RuntimeError("GITHUB_TOKEN이 필요합니다.")

# --- Repo 클론 또는 업데이트 ---
repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"

if not os.path.exists(PROJECT_DIR):
    print(f"'{GITHUB_REPO}' 클론 중...")
    result = subprocess.run(
        ["git", "clone", "--branch", GITHUB_BRANCH, repo_url, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        err = result.stderr.replace(GITHUB_TOKEN, "***")
        raise RuntimeError(f"클론 실패:\n{err}")
    print(f"✅ Repo 클론 완료: {PROJECT_DIR}")
else:
    print(f"ℹ️  이미 클론됨 ({PROJECT_DIR}) — 최신으로 업데이트합니다.")
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "remote", "set-url", "origin", repo_url],
        capture_output=True
    )
    result = subprocess.run(
        ["git", "-C", PROJECT_DIR, "pull"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or "이미 최신 상태입니다.")

# --- 작업 디렉토리 이동 ---
%cd {PROJECT_DIR}
print(f"\n작업 디렉토리: {os.getcwd()}")

# --- 패키지 설치 ---
print("\n패키지 설치 중...")
!pip install -q -r requirements.txt
print("✅ 패키지 설치 완료")

## ④ 설정 확인

> 아래 표를 확인하고 설정이 맞으면 다음 셀로 진행하세요.  
> 변경이 필요하면 `config.yaml`을 직접 수정 후 이 셀을 다시 실행하세요.  
> **Tune 관련 파라미터**: `tune.iterations` (기본 30회), `tune.epochs` (반복당 에포크, 기본 30)

In [ ]:
# ============================================================
# ④ config.yaml → env=colab 패치 및 설정 출력
#    (run_tune.bat의 show_config.py + env 전환 역할)
# ============================================================
import yaml, sys, os
sys.path.insert(0, "src")

with open("config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# env를 colab으로 전환
cfg["env"] = "colab"
with open("config.yaml", "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, allow_unicode=True)

# 설정 출력 (show_config.py와 동일한 테이블 형식)
def print_dict(d, prefix=''):
    for k, v in d.items():
        if isinstance(v, dict):
            print_dict(v, prefix + k + '.')
        else:
            v_str = ', '.join(map(str, v)) if isinstance(v, list) else str(v)
            print(f"| {prefix+k:<33} | {v_str:<25} |")

print("+" + "-"*35 + "+" + "-"*27 + "+")
print(f"| {'Parameter':<33} | {'Value':<25} |")
print("+" + "-"*35 + "+" + "-"*27 + "+")
print_dict(cfg)
print("+" + "-"*35 + "+" + "-"*27 + "+")
print()

# Tune 파라미터 별도 강조 출력
tc = cfg.get("tune", {})
iterations = tc.get('iterations', 30)
epochs     = tc.get('epochs', 30)
total_est  = iterations * epochs
print(f"🔖 Tune 설정: iterations={iterations}회 × epochs={epochs} = 총 ~{total_est} 에포크")
print(f"   모델: {tc.get('model', 'weights/yolov8n.pt')} | imgsz: {tc.get('imgsz', 640)}")
print()
print("⚠️  위 설정을 확인 후, 문제없으면 다음 셀(⑤ 전처리)을 실행하세요.")

## ⑤ 전처리

- `preprocessed_data/` 폴더가 이미 있으면 **자동으로 건너뜁니다**
- 강제 재처리: `FORCE_PREPROCESS = True` 로 변경 후 실행

> **사전 준비**: Drive `pcb-defect-inspection/dataset.zip` 업로드 필요

In [ ]:
# ============================================================
# ⑤ 데이터 전처리 (run_tune.bat의 preprocess 선택 단계)
#    preprocessed_data/ 가 있으면 스킵, 없으면 Drive에서 zip 복사 후 실행
# ============================================================
import sys, os, shutil
from pathlib import Path
sys.path.insert(0, "src")
from utils import load_config, get_paths

FORCE_PREPROCESS = False  # True로 변경하면 강제 재처리

cfg = load_config("config.yaml")
paths = get_paths(cfg)
processed_dir = paths["processed"]
train_img_dir = processed_dir / "images" / "train"

already_done = train_img_dir.exists() and any(train_img_dir.glob("*.jpg"))

if already_done and not FORCE_PREPROCESS:
    n_train = len(list((processed_dir / "images" / "train").glob("*.jpg")))
    n_val   = len(list((processed_dir / "images" / "val").glob("*.jpg")))
    n_test  = len(list((processed_dir / "images" / "test").glob("*.jpg")))
    print("ℹ️  이미 전처리된 데이터가 있습니다 — 건너뜁니다.")
    print(f"   train: {n_train}장 / val: {n_val}장 / test: {n_test}장")
    print("   강제 재처리: FORCE_PREPROCESS = True 로 변경 후 재실행")
else:
    # --- dataset.zip 준비 ---
    local_zip = Path(PROJECT_DIR) / "dataset.zip"
    if not local_zip.exists():
        if os.path.exists(DRIVE_DATASET_ZIP):
            size_mb = os.path.getsize(DRIVE_DATASET_ZIP) / 1e6
            print(f"Drive에서 dataset.zip 복사 중... ({size_mb:.1f} MB)")
            shutil.copy2(DRIVE_DATASET_ZIP, local_zip)
            print("✅ 복사 완료")
        else:
            msg = (
                "dataset.zip을 찾을 수 없습니다.\n"
                f"Drive 경로: {DRIVE_DATASET_ZIP}\n"
                f"Local 경로: {local_zip}\n"
                "Google Drive의 pcb-defect-inspection/ 폴더에 dataset.zip을 업로드하세요."
            )
            raise FileNotFoundError(msg)
    else:
        size_mb = local_zip.stat().st_size / 1e6
        print(f"ℹ️  dataset.zip 로컬에 존재 ({size_mb:.1f} MB)")

    # --- 전처리 실행 ---
    print("\n전처리 시작...")
    !python src/preprocess.py --config config.yaml
    print("\n✅ 전처리 완료")

## ⑥ 이전 튜닝 결과 체크

Drive의 `runs/tune` 폴더를 자동으로 감지하고, Drive와 실시간 동기화 연결(심볼릭 링크)을 설정합니다.

- **`runs/tune` 없음** → 새 튜닝 시작
- **`runs/tune` 있음** → 기존 결과를 덮어쓸지 여부를 선택 (`OVERWRITE_TUNE`)

| 변수 | 동작 |
|------|------|
| `OVERWRITE_TUNE = None` | **자동 감지** (기본값, 권장) |
| `OVERWRITE_TUNE = True` | 이전 결과 무시하고 새 튜닝 시작 |
| `OVERWRITE_TUNE = False` | 이전 결과 있으면 튜닝 취소 |

> **주의**: YOLO tune은 `exist_ok=True`로 실행되므로 이전 결과가 있어도 새 튜닝을 이어서 진행합니다.

In [ ]:
# ============================================================
# ⑥ 이전 튜닝 결과 체크 & Drive 실시간 동기화 설정
#    runs 폴더를 Google Drive에 심볼릭 링크로 연결하여
#    튜닝 중 세션이 끊겨도 결과가 Drive에 실시간 저장되도록 함.
# ============================================================
import os, shutil
from pathlib import Path

# None=자동 감지, True=이전 결과 무시 새 튜닝, False=이전 결과 있으면 취소
OVERWRITE_TUNE = None

DRIVE_RUNS = f"{DRIVE_PROJECT_DIR}/runs"
LOCAL_RUNS = f"{PROJECT_DIR}/runs"

# 1. Drive 실시간 저장을 위한 심볼릭 링크(Symlink) 설정
os.makedirs(DRIVE_RUNS, exist_ok=True)
if os.path.exists(LOCAL_RUNS) and not os.path.islink(LOCAL_RUNS):
    shutil.rmtree(LOCAL_RUNS)  # 기존 로컬 폴더 제거
if not os.path.exists(LOCAL_RUNS):
    os.symlink(DRIVE_RUNS, LOCAL_RUNS)  # 로컬 runs -> Drive runs 연결
print("[Drive 동기화] 로컬 runs/ 폴더가 Google Drive에 실시간 연결되었습니다.")

# 2. 이전 튜닝 결과 확인
DRIVE_TUNE_DIR = Path(DRIVE_RUNS) / "tune"
tune_exists = DRIVE_TUNE_DIR.exists() and any(DRIVE_TUNE_DIR.iterdir())

print()
if tune_exists:
    best_hp = DRIVE_TUNE_DIR / "best_hyperparameters.yaml"
    print(f"⚠️  이전 튜닝 결과 발견: {DRIVE_TUNE_DIR}")
    if best_hp.exists():
        print(f"   best_hyperparameters.yaml ✅ 존재")
    else:
        print(f"   best_hyperparameters.yaml ❌ 미생성 (튜닝 중단된 것으로 보임)")
else:
    print("ℹ️  이전 튜닝 결과 없음 → 새 튜닝을 시작합니다.")

# 3. 실행 여부 결정
if OVERWRITE_TUNE is None:
    PROCEED_TUNE = True  # exist_ok=True이므로 항상 진행
    if tune_exists:
        print("   → exist_ok=True로 설정되어 있어 이어서 진행합니다.")
        print("   → 완전히 새로 시작하려면: OVERWRITE_TUNE = True로 변경 후 재실행")
elif OVERWRITE_TUNE:
    PROCEED_TUNE = True
    print("OVERWRITE_TUNE = True → 이전 결과 무시하고 새 튜닝 시작")
else:  # OVERWRITE_TUNE = False
    if tune_exists:
        PROCEED_TUNE = False
        print("OVERWRITE_TUNE = False → 이전 결과가 있어 튜닝을 취소합니다.")
    else:
        PROCEED_TUNE = True

print()
print("=" * 42)
print(f"  PROCEED_TUNE = {PROCEED_TUNE}")
if PROCEED_TUNE:
    print("  🚀 다음 셀: 하이퍼파라미터 튜닝 시작")
else:
    print("  ⏹️  튜닝이 취소되었습니다.")
    print("  실행하려면: OVERWRITE_TUNE = True로 변경 후 재실행")
print("=" * 42)

## ⑦ 튜닝 실행

> ⑥ 셀의 `PROCEED_TUNE` 값이 `True`일 때만 실행됩니다.  
> 튜닝 결과는 `runs/tune/` 폴더에 저장되며, Drive와 실시간 동기화됩니다.  
>
> **저장 경로**:
> - 튜닝 결과 전체: `runs/tune/`
> - 최적 하이퍼파라미터: `runs/tune/best_hyperparameters.yaml`

In [ ]:
# ============================================================
# ⑦ 하이퍼파라미터 튜닝 실행 (run_tune.bat의 python src/tune.py 역할)
# ============================================================
if not PROCEED_TUNE:
    print("⏹️  PROCEED_TUNE = False — 튜닝이 취소되었습니다.")
    print("   ⑥ 셀에서 OVERWRITE_TUNE = True로 변경 후 다시 실행하세요.")
else:
    import yaml
    # Tune 파라미터 요약 출력
    with open("config.yaml", "r", encoding="utf-8") as f:
        _cfg = yaml.safe_load(f)
    tc = _cfg.get("tune", {})
    iterations = tc.get('iterations', 30)
    epochs     = tc.get('epochs', 30)

    print("=" * 52)
    print("  🚀 하이퍼파라미터 튜닝 시작")
    print(f"  iterations: {iterations}회 | epochs/iter: {epochs}")
    print(f"  총 예상 에포크: ~{iterations * epochs}")
    print("  결과: runs/tune/best_hyperparameters.yaml")
    print("=" * 52)
    print()

    !python src/tune.py --config config.yaml

    print()
    print("=" * 52)
    print("  튜닝 종료 (또는 세션 중단)")
    print("  → 반드시 다음 셀 ⑧ Drive 동기화를 실행하세요!")
    print("=" * 52)

## ⑧ Drive 동기화 (선택/이중안전)

> ⑥ 셀에서 심볼릭 링크를 설정했다면 이 셀을 잊어도 **튜닝 결과는 Drive에 안전**합니다.  
> 
> 다만 최종적으로 `weights/` 폴더를 복사해두기 위해 실행해두면 좋습니다.

In [ ]:
# ============================================================
# ⑧ Drive 동기화 (이중안전)
# ============================================================
import shutil, os
from pathlib import Path

def sync_dir_to_drive(local_dir: str, drive_dir: str) -> int:
    local_path = Path(local_dir)
    drive_path = Path(drive_dir)
    if not local_path.exists():
        print(f"  ⚠️  없음 (건너뜀): {local_dir}")
        return 0

    # If local_dir is a symlink to drive_dir, skip copying files within it
    # as they are already synchronized.
    if local_path.is_symlink() and local_path.resolve() == drive_path.resolve():
        print(f"  ℹ️  '{local_dir}'는 이미 Drive에 심볼릭 링크되어 있습니다 — 복사 건너뜀.")
        return 0

    drive_path.mkdir(parents=True, exist_ok=True)
    count = 0
    for src in local_path.rglob("*"):
        if src.is_file():
            dst = drive_path / src.relative_to(local_path)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            count += 1
    return count

print("Drive 동기화 중...\n")

# runs/ 동기화 (runs/tune/ 폴더 포함)
n1 = sync_dir_to_drive(
    f"{PROJECT_DIR}/runs",
    f"{DRIVE_PROJECT_DIR}/runs"
)
print(f"  ✅ runs/    → Drive ({n1}개 파일)")

# weights/ 동기화
n2 = sync_dir_to_drive(
    f"{PROJECT_DIR}/weights",
    f"{DRIVE_PROJECT_DIR}/weights"
)
print(f"  ✅ weights/ → Drive ({n2}개 파일)")

print()
print(f"🎉 동기화 완료 → {DRIVE_PROJECT_DIR}")
print("   최적 하이퍼파라미터: runs/tune/best_hyperparameters.yaml")

## ⑨ 결과 확인 (선택 사항)

튜닝이 완료된 후 최적 하이퍼파라미터와 튜닝 결과 그래프를 확인합니다.

In [ ]:
# ============================================================
# ⑨ 튜닝 결과 확인 (선택)
# ============================================================
from IPython.display import Image, display
from pathlib import Path
import yaml, sys
sys.path.insert(0, "src")
from utils import load_config, get_paths

cfg = load_config("config.yaml")
paths = get_paths(cfg)
tune_dir = paths["runs"] / "tune"

# --- 최적 하이퍼파라미터 출력 ---
best_hp_path = tune_dir / "best_hyperparameters.yaml"
print("=" * 52)
print("  📋 최적 하이퍼파라미터 (best_hyperparameters.yaml)")
print("=" * 52)
if best_hp_path.exists():
    with open(best_hp_path, "r", encoding="utf-8") as f:
        best_hp = yaml.safe_load(f)
    for k, v in best_hp.items():
        print(f"  {k:<20}: {v}")
    print()
    print(f"📁 파일 위치: {best_hp_path}")
else:
    print(f"  ❌ 파일 없음: {best_hp_path}")
    print("  튜닝이 완료되지 않았거나 결과 경로를 확인하세요.")
print("=" * 52)
print()

# --- 튜닝 결과 이미지 출력 ---
plots = [
    ("tune_results.png", "📊 튜닝 수렴 결과"),
    ("tune_scatter_plots.png", "📊 파라미터 산점도"),
]

found = False
for filename, title in plots:
    p = tune_dir / filename
    if p.exists():
        print(title)
        display(Image(str(p)))
        found = True

if not found:
    # Ultralytics 버전에 따라 파일명이 다를 수 있으므로 전체 목록 출력
    if tune_dir.exists():
        pngs = list(tune_dir.glob("*.png"))
        if pngs:
            print("📂 tune 디렉토리의 이미지 파일:")
            for p in pngs:
                print(f"  {p.name}")
                display(Image(str(p)))
        else:
            print(f"결과 이미지 없음: {tune_dir}")
    else:
        print(f"결과 폴더 없음: {tune_dir}")
        print("튜닝이 완료되지 않았거나 결과 경로를 확인하세요.")